<a href="https://colab.research.google.com/github/rezar362/Portfolio/blob/main/10k-rag/RAG_10K.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Re-download with 500k chars instead of 50k per filing

import requests
import time
import os

DRIVE_PATH  = '/content/drive/MyDrive/rag_10k'
FILINGS_DIR = f'{DRIVE_PATH}/filings'

COMPANIES = {
    'AAPL':'Apple','MSFT':'Microsoft','NVDA':'NVIDIA','GOOGL':'Alphabet',
    'AMZN':'Amazon','META':'Meta','TSLA':'Tesla','AVGO':'Broadcom',
    'COST':'Costco','AMD':'AMD','NFLX':'Netflix','ADBE':'Adobe',
    'QCOM':'Qualcomm','INTU':'Intuit','TXN':'Texas Instruments',
    'ISRG':'Intuitive Surgical','BKNG':'Booking Holdings',
    'AMAT':'Applied Materials','MU':'Micron','LRCX':'Lam Research'
}

def get_cik(ticker):
    headers = {'User-Agent': 'Reza Roshani reza@example.com'}
    resp = requests.get("https://www.sec.gov/files/company_tickers.json", headers=headers)
    data = resp.json()
    for key, val in data.items():
        if val['ticker'].upper() == ticker.upper():
            return str(val['cik_str']).zfill(10)
    return None

def get_10k_text(cik, ticker):
    headers = {'User-Agent': 'Reza Roshani reza@example.com'}
    url  = f"https://data.sec.gov/submissions/CIK{cik}.json"
    resp = requests.get(url, headers=headers)
    data = resp.json()

    filings = data['filings']['recent']
    forms   = filings['form']
    dates   = filings['filingDate']
    accnums = filings['accessionNumber']
    docs    = filings['primaryDocument']

    for i, form in enumerate(forms):
        if form == '10-K' and dates[i] >= '2023-01-01':
            accnum   = accnums[i].replace('-', '')
            doc_name = docs[i]
            text_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accnum}/{doc_name}"
            text_resp = requests.get(text_url, headers=headers)
            # FIXED: 500k chars instead of 50k
            return text_resp.text[:500000], dates[i]
    return None, None

print("── Re-downloading 10-K filings (500k chars) ──")

for ticker in COMPANIES:
    filepath = f'{FILINGS_DIR}/{ticker}_10K.txt'
    # Force re-download by deleting old file
    if os.path.exists(filepath):
        os.remove(filepath)

    cik = get_cik(ticker)
    if not cik:
        print(f"   {ticker:6s} ✗ CIK not found")
        continue

    text, date = get_10k_text(cik, ticker)
    if text:
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(text)
        print(f"   {ticker:6s} ✓ {date} ({len(text):,} chars)")
    else:
        print(f"   {ticker:6s} ✗ not found")

    time.sleep(2)

print("\n✅ Re-download complete — now rerun Block 2")

── Re-downloading 10-K filings (500k chars) ──
   AAPL   ✓ 2025-10-31 (500,000 chars)
   MSFT   ✓ 2025-07-30 (500,000 chars)
   NVDA   ✓ 2026-02-25 (500,000 chars)
   GOOGL  ✓ 2026-02-05 (500,000 chars)
   AMZN   ✓ 2026-02-06 (500,000 chars)
   META   ✓ 2026-01-29 (500,000 chars)
   TSLA   ✓ 2026-01-29 (500,000 chars)
   AVGO   ✓ 2025-12-18 (500,000 chars)
   COST   ✓ 2025-10-08 (500,000 chars)
   AMD    ✓ 2026-02-04 (500,000 chars)
   NFLX   ✓ 2026-01-23 (500,000 chars)
   ADBE   ✓ 2026-01-15 (500,000 chars)
   QCOM   ✓ 2025-11-05 (500,000 chars)
   INTU   ✓ 2025-09-03 (500,000 chars)
   TXN    ✓ 2026-02-06 (500,000 chars)
   ISRG   ✓ 2026-02-03 (500,000 chars)
   BKNG   ✓ 2026-02-18 (500,000 chars)
   AMAT   ✓ 2025-12-12 (500,000 chars)
   MU     ✓ 2025-10-03 (500,000 chars)
   LRCX   ✓ 2025-08-11 (500,000 chars)

✅ Re-download complete — now rerun Block 2


In [5]:
# Block 2 (fixed): Smaller chunk size to get more chunks per filing.
# Previous run had chunk_size=500 words which was too large for 50k char files.
# Now using chunk_size=150 words with overlap=20 for better retrieval granularity.

import os
import re
from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer

DRIVE_PATH  = '/content/drive/MyDrive/rag_10k'
FILINGS_DIR = f'{DRIVE_PATH}/filings'
CHROMA_DIR  = f'{DRIVE_PATH}/chroma_db'

COMPANIES = {
    'AAPL':'Apple','MSFT':'Microsoft','NVDA':'NVIDIA','GOOGL':'Alphabet',
    'AMZN':'Amazon','META':'Meta','TSLA':'Tesla','AVGO':'Broadcom',
    'COST':'Costco','AMD':'AMD','NFLX':'Netflix','ADBE':'Adobe',
    'QCOM':'Qualcomm','INTU':'Intuit','TXN':'Texas Instruments',
    'ISRG':'Intuitive Surgical','BKNG':'Booking Holdings',
    'AMAT':'Applied Materials','MU':'Micron','LRCX':'Lam Research'
}

# ── 1. Load embedding model ───────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding model loaded")

# ── 2. Text cleaning ─────────────────────────────────────────
def clean_text(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s\.\,\;\:\-\(\)\%\$]', ' ', text)
    return text.strip()

# FIXED: chunk_size=150 words instead of 500 — gets 30-50 chunks per filing
def chunk_text(text, chunk_size=150, overlap=20):
    words  = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if len(chunk) > 50:
            chunks.append(chunk)
    return chunks

# ── 3. Reset and reinitialize ChromaDB ───────────────────────
# Delete old collection with wrong chunk sizes
client = PersistentClient(path=CHROMA_DIR)
try:
    client.delete_collection("sec_10k")
    print("✅ Old collection deleted")
except:
    pass

collection = client.get_or_create_collection(
    name="sec_10k",
    metadata={"hnsw:space": "cosine"}
)
print(f"✅ ChromaDB reinitialized")

# ── 4. Embed and store ────────────────────────────────────────
print("\n── Embedding filings ──")
total_chunks = 0

for ticker, company in COMPANIES.items():
    filepath = f'{FILINGS_DIR}/{ticker}_10K.txt'
    if not os.path.exists(filepath):
        print(f"   {ticker:6s} ✗ file not found")
        continue

    with open(filepath, 'r', encoding='utf-8') as f:
        raw = f.read()

    text   = clean_text(raw)
    chunks = chunk_text(text, chunk_size=150, overlap=20)

    embeddings = embedder.encode(chunks, show_progress_bar=False).tolist()

    ids       = [f"{ticker}_{i}" for i in range(len(chunks))]
    metadatas = [{"ticker": ticker, "company": company, "chunk_id": i}
                 for i in range(len(chunks))]

    collection.add(
        ids        = ids,
        embeddings = embeddings,
        documents  = chunks,
        metadatas  = metadatas
    )

    total_chunks += len(chunks)
    print(f"   {ticker:6s} ✓ {len(chunks)} chunks embedded")

print(f"\n✅ Total chunks in DB : {total_chunks}")
print(f"✅ Collection size    : {collection.count()}")
print(f"✅ Avg chunks/company : {total_chunks // len(COMPANIES)}")
print("✅ Block 2 complete")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded
✅ Old collection deleted
✅ ChromaDB reinitialized

── Embedding filings ──
   AAPL   ✓ 156 chunks embedded
   MSFT   ✓ 53 chunks embedded
   NVDA   ✓ 219 chunks embedded
   GOOGL  ✓ 155 chunks embedded
   AMZN   ✓ 142 chunks embedded
   META   ✓ 232 chunks embedded
   TSLA   ✓ 189 chunks embedded
   AVGO   ✓ 155 chunks embedded
   COST   ✓ 142 chunks embedded
   AMD    ✓ 191 chunks embedded
   NFLX   ✓ 142 chunks embedded
   ADBE   ✓ 185 chunks embedded
   QCOM   ✓ 251 chunks embedded
   INTU   ✓ 149 chunks embedded
   TXN    ✓ 113 chunks embedded
   ISRG   ✓ 208 chunks embedded
   BKNG   ✓ 159 chunks embedded
   AMAT   ✓ 156 chunks embedded
   MU     ✓ 148 chunks embedded
   LRCX   ✓ 189 chunks embedded

✅ Total chunks in DB : 3334
✅ Collection size    : 3334
✅ Avg chunks/company : 166
✅ Block 2 complete


In [13]:
# Block 3: RAG pipeline using ChromaDB retrieval + Llama 3 via Groq API.
# Groq is free and much faster than HuggingFace inference.

import os
import requests
from google.colab import userdata
from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer

DRIVE_PATH = '/content/drive/MyDrive/rag_10k'
CHROMA_DIR = f'{DRIVE_PATH}/chroma_db'
GROQ_KEY   = userdata.get('10K_Grok')

COMPANIES = {
    'AAPL':'Apple','MSFT':'Microsoft','NVDA':'NVIDIA','GOOGL':'Alphabet',
    'AMZN':'Amazon','META':'Meta','TSLA':'Tesla','AVGO':'Broadcom',
    'COST':'Costco','AMD':'AMD','NFLX':'Netflix','ADBE':'Adobe',
    'QCOM':'Qualcomm','INTU':'Intuit','TXN':'Texas Instruments',
    'ISRG':'Intuitive Surgical','BKNG':'Booking Holdings',
    'AMAT':'Applied Materials','MU':'Micron','LRCX':'Lam Research'
}

# ── 1. Load ChromaDB and embedder ────────────────────────────
client     = PersistentClient(path=CHROMA_DIR)
collection = client.get_collection("sec_10k")
embedder   = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✅ ChromaDB loaded: {collection.count()} chunks")

# ── 2. Retrieval function ─────────────────────────────────────
def retrieve(query, ticker=None, n_results=3):
    query_embedding = embedder.encode([query]).tolist()
    where   = {"ticker": ticker} if ticker else None
    results = collection.query(
        query_embeddings = query_embedding,
        n_results        = n_results,
        where            = where
    )
    return results['documents'][0], results['metadatas'][0]

# ── 3. Groq generation function ───────────────────────────────
def generate(prompt, max_tokens=512):
    headers = {
        "Authorization": f"Bearer {GROQ_KEY}",
        "Content-Type" : "application/json"
    }
    payload = {
        "model"    : "llama-3.1-8b-instant",
        "messages" : [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.3
    }
    resp   = requests.post("https://api.groq.com/openai/v1/chat/completions",
                           headers=headers, json=payload)
    result = resp.json()
    if 'choices' in result:
        return result['choices'][0]['message']['content'].strip()
    return f"Error: {result}"

# ── 4. RAG query function ─────────────────────────────────────
def rag_query(question, ticker=None, n_results=5):
    chunks, metas = retrieve(question, ticker=ticker, n_results=n_results)

    context = ""
    for chunk, meta in zip(chunks, metas):
        context += f"\n[{meta['company']} — {meta['ticker']}]\n{chunk}\n"

    company_filter = f"Focus on {COMPANIES.get(ticker, ticker)}." if ticker else "Use all available companies."

    prompt = f"""You are a financial analyst assistant. Answer the question based ONLY on the provided SEC 10-K filing excerpts. Be concise and specific. {company_filter}

Context from SEC 10-K filings:
{context}

Question: {question}

Answer:"""

    answer = generate(prompt)
    return answer, metas

# ── 5. Test queries ───────────────────────────────────────────
print("\n── Test Queries ──\n")

queries = [
    ("What are the main risk factors for Apple?",    "AAPL"),
    ("How did NVIDIA describe its revenue growth?",  "NVDA"),
    ("What is Microsoft's cloud strategy?",          "MSFT"),
    ("Compare risk factors between Apple and Tesla", None),
]

for question, ticker in queries:
    print(f"Q: {question}")
    answer, metas = rag_query(question, ticker=ticker)
    print(f"A: {answer[:500]}")
    print(f"   Sources: {list(set(m['ticker'] for m in metas))}")
    print()

print("✅ Block 3 complete")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ ChromaDB loaded: 3334 chunks

── Test Queries ──

Q: What are the main risk factors for Apple?
A: Based on the provided SEC 10-K filing excerpts, the main risk factors for Apple include:

1. Supply shortages and price increases that can affect its business, results of operations, financial condition, and stock price.
2. Design and manufacturing defects that can harm its reputation and result in financial losses.
3. Major public health issues, such as pandemics, that can impact the global economy and demand for consumer products.
4. Disruptions in operations, supply chain, and sales and distr
   Sources: ['AAPL']

Q: How did NVIDIA describe its revenue growth?
A: NVIDIA did not describe its revenue growth in the provided excerpts. The excerpts discuss potential risks and uncertainties that could impact NVIDIA's revenue, expenses, and development timelines, but they do not provide information on actual revenue growth.
   Sources: ['NVDA']

Q: What is Microsoft's cloud strategy?
A: Base

In [14]:
# Block 4: Interactive Q&A interface for the 10-K RAG system.
# Supports single company queries, cross-company comparisons,
# and a predefined set of analyst-style questions.

import time
from IPython.display import display, HTML

COMPANIES = {
    'AAPL':'Apple','MSFT':'Microsoft','NVDA':'NVIDIA','GOOGL':'Alphabet',
    'AMZN':'Amazon','META':'Meta','TSLA':'Tesla','AVGO':'Broadcom',
    'COST':'Costco','AMD':'AMD','NFLX':'Netflix','ADBE':'Adobe',
    'QCOM':'Qualcomm','INTU':'Intuit','TXN':'Texas Instruments',
    'ISRG':'Intuitive Surgical','BKNG':'Booking Holdings',
    'AMAT':'Applied Materials','MU':'Micron','LRCX':'Lam Research'
}

# ── 1. Analyst question presets ───────────────────────────────
PRESET_QUESTIONS = {
    '1': ('What are the main risk factors?',              None),
    '2': ('What is the revenue growth strategy?',         None),
    '3': ('How does the company describe competition?',   None),
    '4': ('What are the key products and services?',      None),
    '5': ('What macroeconomic risks does the company face?', None),
    '6': ('Compare AI strategy across NVIDIA, Microsoft and Google', None),
    '7': ('Which companies mention supply chain risks?',  None),
    '8': ('Compare revenue growth across Apple, Amazon and Meta', None),
}

# ── 2. Pretty print function ──────────────────────────────────
def print_answer(question, answer, metas, ticker=None):
    sources = list(set(m['ticker'] for m in metas))
    print("─" * 60)
    print(f"Q: {question}")
    if ticker:
        print(f"   Company: {COMPANIES.get(ticker, ticker)} ({ticker})")
    print(f"   Sources: {sources}")
    print()
    print(f"A: {answer}")
    print()

# ── 3. Preset analyst queries ─────────────────────────────────
print("── Preset Analyst Queries ──\n")

analyst_queries = [
    ("What are the main risk factors for NVIDIA?",              "NVDA"),
    ("How does Apple describe its competition?",                "AAPL"),
    ("What is Amazon's cloud and AI growth strategy?",          "AMZN"),
    ("Compare AI strategy across NVIDIA, Microsoft and Google", None),
    ("Which companies mention supply chain risks?",             None),
    ("Compare revenue growth across Apple, Amazon and Meta",    None),
]

for question, ticker in analyst_queries:
    answer, metas = rag_query(question, ticker=ticker, n_results=3)
    print_answer(question, answer, metas, ticker)
    time.sleep(8)  # stay under Groq rate limit

# ── 4. Save results to Drive ──────────────────────────────────
import json

results = []
for question, ticker in analyst_queries:
    answer, metas = rag_query(question, ticker=ticker, n_results=3)
    results.append({
        'question': question,
        'ticker'  : ticker,
        'answer'  : answer,
        'sources' : list(set(m['ticker'] for m in metas))
    })
    time.sleep(8)

with open(f'{DRIVE_PATH}/rag_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"✅ Results saved to Drive")

# ── 5. Interactive query ──────────────────────────────────────
print("\n── Interactive Query ──")
print("Available tickers:", list(COMPANIES.keys()))
print()

question = input("Enter your question: ")
ticker   = input("Enter ticker (or press Enter for all companies): ").strip().upper()
ticker   = ticker if ticker in COMPANIES else None

answer, metas = rag_query(question, ticker=ticker, n_results=3)
print_answer(question, answer, metas, ticker)

print("✅ Block 4 complete")

── Preset Analyst Queries ──

────────────────────────────────────────────────────────────
Q: What are the main risk factors for NVIDIA?
   Company: NVIDIA (NVDA)
   Sources: ['NVDA']

A: Based on the provided SEC 10-K filing excerpts, the main risk factors for NVIDIA are:

1. Security breaches and unauthorized access to proprietary information or sensitive data.
2. Supply chain risks, including:
   - Lack of guaranteed supply of components and capacity.
   - Decommitment by suppliers.
   - Potential higher wafer and component prices due to incorrectly estimating demand.
   - Failure by foundries or contract manufacturers to procure raw materials or provide adequate manufacturing or test capacity.
   - Failure by foundries to develop, obtain, or successfully implement high-quality process technologies.
   - Failure by suppliers to comply with requirements.

────────────────────────────────────────────────────────────
Q: How does Apple describe its competition?
   Company: Apple (AAPL)
